# Task 5: Mental Health Support Chatbot (Fine-Tuned)

**DevelopersHub Corporation — AI/ML Engineering Internship**

## Problem Statement

Many people hesitate to talk about everyday stress, anxiety, or low mood,
and a gentle, always-available listener can help them feel heard before (or
alongside) seeking professional support. The goal of this notebook is to
fine-tune a small open-source language model so it responds with empathy and
emotional warmth, in the style of the **EmpatheticDialogues** dataset.

## Objective

- Fine-tune `distilgpt2` on the EmpatheticDialogues dataset using
  Hugging Face's `Trainer` API.
- Produce a model that responds gently and supportively to statements about
  stress, anxiety, sadness, or loneliness.
- Wrap the model in a safety-aware chat interface (CLI + Streamlit) that
  **never** tries to handle a real mental-health crisis itself — it
  recognizes crisis language and redirects to real human help instead.

## Model Base

`distilgpt2` (82M parameters) — small enough to fine-tune quickly on a single
GPU, while still being a capable causal language model.

## Dataset

**EmpatheticDialogues** (Facebook AI) — ~25k crowd-sourced conversations
grounded in a speaker's emotional situation, each paired with an empathetic
listener response. Loaded directly from the Hugging Face `datasets` hub.

## ⚠️ A note on running this notebook

This notebook was developed in a sandboxed environment whose network access
is restricted to a short allow-list of domains — it **cannot reach
`huggingface.co`** to download the base model or the dataset, and has no GPU.
The fine-tuning code below is the complete, real implementation and is meant
to be run on an environment with internet + a GPU, such as **Google Colab or
Kaggle Notebooks (both free)**.

To make this notebook still runnable and demonstrable end-to-end as-is, each
network-dependent step is wrapped so it **fails gracefully with a clear
message** here, and a lightweight **offline rule-based fallback responder**
is provided so the conversational interface (Section 6 onward) can be fully
tested without internet or a GPU. Swap `HF_HUB_AVAILABLE` to `True` (i.e. run
this on Colab) to use the real fine-tuned model instead.


## 1. Imports & Environment Check

In [1]:
import os
import re
import random

import torch

print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())


PyTorch version: 2.12.0+cu130
CUDA (GPU) available: False


## 2. Load Base Model & Dataset

This step requires internet access to the Hugging Face Hub. It is wrapped in
a `try/except` so the notebook continues gracefully (using the offline demo
mode) wherever the Hub isn't reachable.

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"
HF_HUB_AVAILABLE = True
tokenizer = None
model = None

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
    print(f"Loaded base model '{MODEL_NAME}' successfully.")
except Exception as e:
    HF_HUB_AVAILABLE = False
    print("Could not reach the Hugging Face Hub from this environment.")
    print(f"  Reason: {e}")
    print("This is expected in a network-restricted sandbox. Run this notebook")
    print("on Google Colab or Kaggle (free GPU + full internet) to fine-tune for real.")


Could not reach the Hugging Face Hub from this environment.
  Reason: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
This is expected in a network-restricted sandbox. Run this notebook
on Google Colab or Kaggle (free GPU + full internet) to fine-tune for real.


In [3]:
from datasets import load_dataset

raw_dataset = None

if HF_HUB_AVAILABLE:
    try:
        raw_dataset = load_dataset("empathetic_dialogues")
        print(raw_dataset)
    except Exception as e:
        HF_HUB_AVAILABLE = False
        print("Could not download the EmpatheticDialogues dataset.")
        print(f"  Reason: {e}")


## 3. Preprocessing

Each EmpatheticDialogues example is formatted as a single training string:

```
Context: <the situation / previous utterance>
Response: <the empathetic listener reply><eos>
```

This teaches the model to continue a `Context:` prompt with an empathetic
`Response:`, which is exactly the pattern we use at inference time.

In [4]:
def format_example(context: str, response: str) -> str:
    return f"Context: {context}\nResponse: {response}{tokenizer.eos_token}"


def tokenize_function(examples):
    texts = [format_example(c, r) for c, r in zip(examples["context"], examples["utterance"])]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=128)


if HF_HUB_AVAILABLE:
    tokenized_dataset = raw_dataset["train"].map(
        tokenize_function, batched=True,
        remove_columns=raw_dataset["train"].column_names,
    )
    print(tokenized_dataset)
else:
    print("Skipping tokenization — base model/dataset weren't available in this environment.")


Skipping tokenization — base model/dataset weren't available in this environment.


## 4. Fine-Tuning with the Hugging Face `Trainer` API

In [5]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "./mental-health-chatbot"

if HF_HUB_AVAILABLE:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        save_strategy="epoch",
        logging_steps=50,
        learning_rate=5e-5,
        report_to="none",
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )

    trainer.train()
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"Fine-tuned model saved to {OUTPUT_DIR}")
else:
    print("Skipping fine-tuning in this environment.")
    print("On Colab/Kaggle with GPU + internet, this cell trains distilgpt2 on")
    print("EmpatheticDialogues for 3 epochs and saves the result to", OUTPUT_DIR)


Skipping fine-tuning in this environment.
On Colab/Kaggle with GPU + internet, this cell trains distilgpt2 on
EmpatheticDialogues for 3 epochs and saves the result to ./mental-health-chatbot


## 5. Crisis Safety Check

Before *any* response is generated — fine-tuned model or offline fallback —
the message is checked for crisis language. If detected, the bot does not
attempt to respond empathetically itself; it immediately redirects to real
human help. This check always runs, regardless of which backend produced
(or would have produced) the reply.

In [6]:
CRISIS_PATTERNS = [
    r"suicide", r"kill myself", r"end my life", r"want to die",
    r"self.?harm", r"hurt myself", r"no reason to live",
]

CRISIS_RESPONSE = (
    "I'm really glad you told me this, and I want to take it seriously: it sounds "
    "like you're going through something very heavy right now. I'm not able to "
    "give you the kind of support you need in this moment, but please reach out "
    "right now to a crisis line or emergency service in your area, or to someone "
    "you trust. You deserve real, immediate support — you don't have to carry this alone."
)

def is_crisis_message(text: str) -> bool:
    t = text.lower()
    return any(re.search(p, t) for p in CRISIS_PATTERNS)


## 6. Response Generation

`generate_with_finetuned_model()` uses the real fine-tuned model (only works
where `HF_HUB_AVAILABLE` is `True`). `offline_demo_response()` is a
lightweight, rule-based fallback so the conversational *interface* can be
fully tested in any environment — including this sandbox.

In [7]:
def generate_with_finetuned_model(user_text: str, max_new_tokens: int = 60) -> str:
    prompt = f"Context: {user_text}\nResponse:"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=True,
        top_p=0.9, temperature=0.8, pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split("Response:")[-1].strip()


_EMPATHETIC_TEMPLATES = {
    "stress":   ["That sounds like a lot to carry right now. What's weighing on you the most?",
                 "Stress like that is exhausting. Have you been able to take any breaks today?"],
    "anxious":  ["It makes sense to feel anxious about that — it matters to you. What's the part that worries you most?",
                 "Anxiety can be really overwhelming. Try taking a slow breath with me — what's on your mind?"],
    "anxiety":  ["It makes sense to feel anxious about that — it matters to you. What's the part that worries you most?"],
    "sad":      ["I'm sorry you're feeling this way. Do you want to tell me more about what's going on?",
                 "That sounds really hard. I'm here to listen, whenever you're ready."],
    "lonely":   ["Feeling lonely is genuinely painful, and I'm glad you reached out instead of sitting with it alone.",
                 "I hear you — feeling unseen by others is hard. I'm here, and I'm listening."],
    "tired":    ["It sounds like you're running on empty. Has it been like this for a while?"],
    "angry":    ["That frustration makes sense given what you described. What happened?"],
    "happy":    ["I love hearing that! Tell me more about what's making you feel good.",
                 "That's wonderful — thank you for sharing something good with me."],
    "great":    ["I love hearing that! Tell me more about what's making you feel good."],
    "excited":  ["That excitement is wonderful — what are you looking forward to most?"],
}

def offline_demo_response(user_text: str) -> str:
    t = user_text.lower()
    for keyword, templates in _EMPATHETIC_TEMPLATES.items():
        if keyword in t:
            return random.choice(templates)
    return "Thank you for sharing that with me. Can you tell me a little more about how you're feeling?"


def get_response(user_text: str) -> str:
    if is_crisis_message(user_text):
        return CRISIS_RESPONSE
    if HF_HUB_AVAILABLE:
        return generate_with_finetuned_model(user_text)
    return offline_demo_response(user_text)


## 7. Testing the Conversational Flow

In [8]:
test_inputs = [
    "I've been feeling really stressed about my exams lately.",
    "I feel so lonely these days, like nobody understands me.",
    "I'm anxious about my job interview tomorrow.",
    "Honestly I've been feeling great this week!",
]

for msg in test_inputs:
    print("You :", msg)
    print("Bot :", get_response(msg))
    print("-" * 80)


You : I've been feeling really stressed about my exams lately.
Bot : That sounds like a lot to carry right now. What's weighing on you the most?
--------------------------------------------------------------------------------
You : I feel so lonely these days, like nobody understands me.
Bot : Feeling lonely is genuinely painful, and I'm glad you reached out instead of sitting with it alone.
--------------------------------------------------------------------------------
You : I'm anxious about my job interview tomorrow.
Bot : It makes sense to feel anxious about that — it matters to you. What's the part that worries you most?
--------------------------------------------------------------------------------
You : Honestly I've been feeling great this week!
Bot : I love hearing that! Tell me more about what's making you feel good.
--------------------------------------------------------------------------------


## 8. Command-Line Chat Interface

Run `run_cli_chat()` in a real terminal (it uses blocking `input()`, so it is
defined here but not auto-executed in the notebook).

In [9]:
def run_cli_chat():
    print("Mental Health Support Chatbot — type 'exit' to quit")
    print("(This is a supportive listening prototype, not a replacement for therapy.)\n")
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() in {"exit", "quit"}:
            print("Bot: Take care of yourself. I'm here whenever you want to talk again.")
            break
        print("Bot:", get_response(user_input))

# Uncomment the line below to chat interactively in a terminal:
# run_cli_chat()


## 9. Streamlit Web Interface

A simple Streamlit app (`app.py`, included alongside this notebook in the
repo) provides a web-based chat interface using the exact same `get_response`
logic. Run it with:

```bash
streamlit run app.py
```


## 10. Conclusion & Insights

- Formatting EmpatheticDialogues as `Context: ... / Response: ...` pairs lets
  even a small model like `distilgpt2` learn a consistent empathetic-reply
  pattern through fine-tuning, without needing a huge model.
- A **safety-first design** matters more than response quality here: the
  crisis-language check runs *before* any generation, on every single
  message, regardless of which backend is active.
- Keeping the chat function (`get_response`) backend-agnostic (fine-tuned
  model vs. offline fallback) made it possible to fully build, test, and
  demo the entire conversational interface even without GPU/internet access.
- **Next steps**: fine-tune a slightly larger base model (e.g. `GPT-Neo-125M`)
  for more natural phrasing, add multi-turn conversation memory, and run a
  small human evaluation of response empathy before any real deployment.

> **Disclaimer:** This chatbot is an educational prototype built for an
> internship task. It is **not** a licensed mental health service and must
> never be used as a substitute for professional therapy, counseling, or
> emergency crisis support.
